In [3]:
import boto3

from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role,
)
from sagemaker.serve import ModelBuilder

REGION = "ap-southeast-2"
ENDPOINT_NAME = "customer-churn-new-endpoint"

aws_session = boto3.Session(region_name=REGION)
sm = aws_session.client("sagemaker")

MODEL_PACKAGE_ARN = (
    "arn:aws:sagemaker:ap-southeast-2:717279701744:"
    "model-package/customer-churn-model-v4/1"
)

package = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

container = package["InferenceSpecification"]["Containers"][0]

model_data_url = container["ModelDataUrl"]
image_uri = container["Image"]

role = get_execution_role()
sagemaker_session = Session(
    boto_session=aws_session
)

print("Model:", MODEL_PACKAGE_ARN)
print("Artifact:", model_data_url)
print("Image:", image_uri)
print("Role:", role)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
Model: arn:aws:sagemaker:ap-southeast-2:717279701744:model-package/customer-churn-model-v4/1
Artifact: s3://sagemaker-ml-lab-717279701744-ap-southeast-2/shared/sagemaker-scikit-learn-job/sagemaker-scikit-learn-job-20261002111504/output/model.tar.gz
Image: 783357654285.dkr.ecr.ap-southeast-2.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3
Role: arn:aws:iam::717279701744:role/service-role/AmazonSageMakerAdminIAMExecutionRole_1


In [4]:
packages = sm.list_model_packages(
    ModelPackageGroupName="customer-churn-model-v4",
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=20,
)["ModelPackageSummaryList"]

approved = [
    item for item in packages
    if item["ModelApprovalStatus"] == "Approved"
    and item["ModelPackageStatus"] == "Completed"
]

if not approved:
    raise RuntimeError("Tidak ada approved model di customer-churn-model-v4")

MODEL_PACKAGE_ARN = approved[0]["ModelPackageArn"]

package = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

container = package["InferenceSpecification"]["Containers"][0]
model_data_url = container["ModelDataUrl"]
image_uri = container["Image"]

print("Using:", MODEL_PACKAGE_ARN)

Using: arn:aws:sagemaker:ap-southeast-2:717279701744:model-package/customer-churn-model-v4/1


In [6]:
model_builder = ModelBuilder(
    s3_model_data_url=model_data_url,
    image_uri=image_uri,
    role_arn=role,
    sagemaker_session=sagemaker_session,
    env_vars={
        "SAGEMAKER_PROGRAM": "inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
    },
)

model = model_builder.build(
    model_name="customer-churn-new-model"
)

endpoint = model_builder.deploy(
    endpoint_name=ENDPOINT_NAME
)

print("Deployment submitted:", ENDPOINT_NAME)

Output()

/miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is 
deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is 
slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources

2026-10-02 13:44:08,051 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)

2026-10-02 13:44:08,055 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)

2026-10-02 13:44:08,056 INFO - sagemaker-containers - nginx config:

...

2026-10-02 13:53:23,800 INFO - sagemaker-containers - No GPUs detected (normal if no gpus installed)

2026-10-02 13:53:23,801 INFO - sagemaker_sklearn_container.serving - Encountered an unexpected error.

[2026-10-02 13:53:23 +0000] [16] [ERROR] Error handling request /ping

Traceback (most recent call last):
  File "/miniconda3/lib/python3.9/site-packages/gunicorn/workers/base_async.py", line 54, in handle
    self.handle_request(listener_name, req, client, addr)
  File "/miniconda3/lib/python3.9/site-packages/gunicorn/workers/ggevent.py", line 127, in handle_request
    super().handle_request(listener_name, req, sock, addr)
  File "/miniconda3/lib/python3.9/site-packages/gunicorn/workers/base_async.py", line 107, in handle_request
    respiter = self.wsgi(environ, resp.start_response)
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/serving.py", line 140, in main
    user_module_transformer, execution_parameters_fn = import_module(serving_env.module_name,
  File "/miniconda3/lib/python3.9/site-packages/sagemaker_sklearn_container/serving.py", line 118, in import_module
    user_module = importlib.import_module(module_name)
  File "/miniconda3/lib/python3.9/importlib/__init__.py", line 127, in import_module
    return _bootstrap._gcd_import(name, package, level)
  File "<frozen importlib._bootstrap>", line 1027, in _gcd_import
  File "<frozen importlib._bootstrap>", line 961, in _sanity_check

Deployment submitted: customer-churn-new-endpoint


In [7]:
import time

while True:
    details = sm.describe_endpoint(
        EndpointName=ENDPOINT_NAME
    )

    status = details["EndpointStatus"]
    print("Endpoint status:", status)

    if status in {"InService", "Failed", "OutOfService"}:
        break

    time.sleep(30)

if status != "InService":
    raise RuntimeError(
        details.get(
            "FailureReason",
            f"Endpoint ended with {status}",
        )
    )

print("Endpoint is InService.")

Endpoint status: InService
Endpoint is InService.
